# RBF SVM: expanded training-only feature selection

This exploratory extension evaluates early versus late elapsed-time stages on held-out participants. It uses the existing pseudonymous feature bundle and the same LOSO/grouped inner validation as the four original models. Labels are proxies for stage, not independently verified physiological fatigue.

The classifier and the feature-count search both change relative to the original linear SVM. An improvement cannot therefore be attributed solely to the RBF kernel. Existing results remain intact. No real-data training has been executed in this delivered notebook.

## Portable setup

This is a clean public source notebook; no participant results are embedded.
Open it from the cloned project. If necessary set `EMG_PROJECT_ROOT` to that
project root. There is no personal Drive path or automatic Drive mount. Mount
your own private storage separately when using Colab. Restore authorized private
inputs as described in the root `README.md` (Dataset and experiment annotations) before running data cells.

Install root `requirements.txt` in the selected environment, restart that kernel,
then run cells in order. The optional installer below defaults to False. CNN
users also need compatible PyTorch; see the root README. Scientific settings,
labels, grouping and model procedures are retained; setup/path handling changes.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from importlib.metadata import version

# Set this only when the notebook is opened outside the cloned project.
PROJECT_ROOT_OVERRIDE = os.environ.get('EMG_PROJECT_ROOT')
start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
PROJECT_ROOT = next((p.resolve() for p in [start, *start.parents]
    if (p / 'project_paths.py').is_file() and (p / 'modelling/core.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the cloned project, or set EMG_PROJECT_ROOT to its root.')
sys.path.insert(0, str(PROJECT_ROOT))
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')])
    raise RuntimeError('Restart this kernel, restore INSTALL_REQUIREMENTS=False, then run from the first cell.')
for package, expected in {'numpy':'2.2.6','pandas':'2.3.2','scipy':'1.17.1','scikit-learn':'1.8.0'}.items():
    actual=version(package)
    print(package,actual)
    if actual!=expected:raise RuntimeError('Install requirements in this environment and restart the kernel.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
probe=subprocess.run([sys.executable,'-c','import numpy, pandas, scipy.sparse, sklearn; print("Fresh-process imports OK")'],capture_output=True,text=True)
print(probe.stdout)
if probe.returncode:raise RuntimeError(probe.stderr)
print('Project:',PROJECT_ROOT)
print('Python:',sys.version.split()[0])


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Image, Markdown
from modelling import core as modelling_core
if not hasattr(modelling_core, "RBF_DEFAULTS"):
    raise RuntimeError("Use core.py from this checkout and restart the kernel.")
from modelling.core import DEFAULTS, RBF_DEFAULTS, MODEL_NAMES, load_data, run_model

MODEL_KEY = "svm_rbf_selected"
DATA_DIR = PROJECT_ROOT / "modelling/data"
OUTPUT_ROOT = PROJECT_ROOT / "modelling/outputs"  # Change to private persistent storage if available.
CONFIG = {**DEFAULTS, **RBF_DEFAULTS}
# Full-cohort refit runs only AFTER the held-out evaluation; its training score is not reported.
CONFIG["final_refit"] = True
assert sklearn.__version__ == "1.8.0", "Install pinned requirements and restart the kernel."

windows, dictionary, provenance = load_data(DATA_DIR)
display(pd.DataFrame([{"model": MODEL_NAMES[MODEL_KEY], "subjects": windows.subject_id.nunique(),
    "sets": windows.recording_id.nunique(), "windows": len(windows),
    "processing_run": provenance["processing_run_id"]}]))
display(pd.Series(CONFIG, name="Frozen choices").to_frame())
print("Original reviewed cohort: 15 subjects, 139 sets, 9215 windows. New reviews may differ.")
print("Configurations per outer fold:", len(CONFIG["C"]) * len(CONFIG["gamma"]) * len(CONFIG["max_features"]))


## 2. Labels, eligibility and class counts

Label **0**: first third of the reviewed active interval. Label **1**: last third. These are **elapsed-time thirds**, adapting the assignment's repetition-count example because repetition markers are unavailable. The middle third and any 500 ms windows crossing a stage boundary are excluded. Hop is 250 ms. Keep both legs/all sets together in every split.

The bundle honors the saved set exclusion (`S009_REC01`), reviewed boundaries and computational/flatline/manual-artifact eligibility rules. Provisional transient/drift flags remain flags rather than automatic exclusions. MVC normalization cannot be performed without MVC references. Preprocessing is offline and can use later samples within the same recording; this does not establish real-time performance.

Earlier whole-cohort statistics and trajectories informed development. Fold-local selection prevents direct fitting on held-out subjects, but does not make this a pristine confirmatory study. The exclusion and boundary reviews also preceded validation. A new cohort would provide a stronger independent test.

### Descriptive class-imbalance audit

Report counts, percentages and the majority/minority ratio overall, by participant, leg and recording. Check every outer training/test partition and every inner training/validation partition. Counts alone need no significance test: overlapping windows are not independent observations. Equal elapsed-time thirds can yield different retained counts after window alignment and eligibility checks. The audit does not change the approved no-resampling/no-class-weighting policy.

The saved class-balance table and figure can be used in the report's dataset/label description. The actual model input remains individual windows in `model_windows.csv`; the 60 rows of `class_counts.csv` are participant × leg × class summaries.

In [ ]:
from modelling.class_balance import audit_balance
BALANCE_OUT = OUTPUT_ROOT / "class_balance" / provenance["data_sha256"][:12]
balance = audit_balance(windows, BALANCE_OUT, seed=CONFIG["seed"], inner_folds=CONFIG["inner_folds"])
for name in ["overall", "by_subject", "by_leg", "loso_partitions"]:
    display(Markdown("**" + name.replace("_", " ") + "**"))
    display(balance[name].round(3))
display(Image(filename=str(BALANCE_OUT / "class_balance.png")))
print((BALANCE_OUT / "class_balance_notes.txt").read_text(encoding="utf-8"))
display(pd.DataFrame({"feature": dictionary.feature,
    "missing_windows": windows[dictionary.feature].isna().sum().to_numpy()}))
display(Markdown("**Excluded set IDs:** " + ", ".join(provenance["excluded_recording_ids"])))

## 3. Approved RBF search and subject-wise evaluation

**Outer:** leave one participant out, 15 folds, keeping both legs and all recordings together. **Inner:** three shuffled subject-grouped folds, seed 20261018. Select the configuration maximizing mean validation-participant macro-F1, weighting participants equally. Each held-out participant is evaluated once.

**Approved feature-count limits:** `[2, 3, 5, 8, 12, 20, 36]`. The selector ranks all 36 candidates using only the current training split: absolute mean participant late-minus-early change divided by sample SD of those changes. Within each set use stage medians with at least three defined windows per stage; average paired set changes within each leg, then legs equally within participant. Both legs and at least three contributing participants are required. Constant/insufficient changes are unavailable. No p-value cutoff or family quota is used.

Walk down this ranking, skipping candidates with absolute Spearman correlation ≥0.90 with an already retained feature on training-participant changes. Unestimable redundancy comparisons are flagged and skipped. The searched limit is a **maximum**, so 36 does not guarantee 36 retained features. This rule examines participant-change redundancy, not every possible nonlinear interaction. Training-window correlations are exported separately.

All selector fitting, median imputation and standard scaling occur inside the Pipeline in every inner training split. Search **C = `[0.1, 1, 10, 100]`** and **gamma = `['scale', 0.01, 0.1, 1]`**. C controls regularization; gamma controls how local the RBF influence is. `scale` uses the current training-transformed inputs, not the held-out participant. Parameter ties follow sklearn's fixed ascending grid order: C, gamma list order, then feature limit.

This gives **112 configurations × 3 inner splits = 336 fits per outer fold**, plus its winning refit. Fifteen outer folds require 5,055 classifier fits; the final all-subject search/refit adds 337, totalling 5,392. Selection audits additionally reconstruct the winning inner selectors without fitting classifiers. Serial execution and a 128 MiB kernel cache bound parallel memory use. Runtime can be substantially longer than linear SVM; no specific completion time is guaranteed.

Use `SVC(kernel='rbf', probability=False, tol=1e-3, max_iter=-1)`; convergence is not capped at the linear solver's iteration limit. No class weighting, resampling, calibration or threshold tuning is introduced. Use decision scores for ROC/PR metrics and default SVC predictions for class metrics. A final grouped search/refit on all subjects produces a usable saved pipeline, not an additional independent test score.

This search was chosen after inspecting the initial models' LOSO results. Preserve those results and disclose the exploratory extension; a new cohort would provide stronger independent confirmation.

## 4. Execute this model

This cell performs the actual training/evaluation. Each completed fold prints progress and saves a fitted pipeline, its split, selected features, winning inner-fold selector audits, search results and warnings. A new timestamped run directory preserves previous runs. A failed run is explicitly marked incomplete and does not replace the latest completed result.

In [ ]:
OUT = run_model(MODEL_KEY, DATA_DIR, OUTPUT_ROOT, config=CONFIG)
print("Completed run:", OUT)
run_manifest = json.loads((OUT / "run_manifest.json").read_text())
assert run_manifest["status"] == "complete"
assert run_manifest["n_outer_folds"] == windows.subject_id.nunique()

## 5. Metrics for Sections 5–6

The primary summary gives each held-out participant equal weight: mean ± **sample SD** across 15 outer folds (`ddof=1`). SD is between-participant variation, not a confidence interval. Macro precision/recall/F1 average the two classes equally; zero denominators use zero and supports are shown. Per-class F1 and Cohen's kappa are included.

ROC-AUC, trapezoidal PR-AUC and **average precision (AP)** use late stage as the positive class. PR-AUC and AP use different integration conventions and are reported separately. RBF SVM uses uncalibrated decision scores; logistic regression and the majority baseline use class-1 probabilities. AUCs are evaluated **within each held-out participant** and summarized across folds, without pooling score scales from different fitted models. Undefined AUCs remain missing and the number of defined folds is shown.

For a constant majority score ROC-AUC is 0.5, AP equals late-stage prevalence, and trapezoidal PR-AUC can be noticeably larger because of endpoint interpolation; that does not indicate discrimination. Pooled window-weighted metrics are secondary. Confusion matrices pool counts across all out-of-fold predictions, so participants with more windows contribute more counts.

In [ ]:
fold_metrics = pd.read_csv(OUT / "results.csv")
summary = pd.read_csv(OUT / "metrics_mean_sd.csv")
summary["mean_plus_minus_sd"] = summary.apply(
    lambda r: f"{r['mean']:.3f} ± {r['sd']:.3f}", axis=1)
display(summary[["metric", "mean_plus_minus_sd", "n_defined", "n_folds"]])
display(fold_metrics[["fold", "test_subject", "accuracy", "precision_macro", "recall_macro",
    "f1_macro", "f1_class_0", "f1_class_1", "cohen_kappa", "roc_auc", "pr_auc", "average_precision"]])
display(pd.read_csv(OUT / "per_class_metrics.csv"))
display(pd.read_csv(OUT / "per_class_mean_sd.csv", header=[0,1], index_col=0))
display(Markdown("**Secondary pooled window metrics:**"))
display(pd.Series(json.loads((OUT / "pooled_window_metrics_secondary.json").read_text())).to_frame("value"))

In [ ]:
for image_name in ["confusion_matrices.png", "roc_pr_by_subject.png",
                   "per_subject_f1.png", "metrics_mean_sd.png", "validation_diagram.png"]:
    display(Image(filename=str(OUT / "figures" / image_name)))

## 6. Selection stability, nonlinear model details and bilateral results

Selection frequency records how often a feature survives the full training-only procedure. It is not a significance test or a measure of independent predictive contribution. RBF SVM does not have one linear coefficient per original feature; no linear coefficient CSV is fabricated. Each fold instead saves kernel parameters, effective gamma and support-vector counts in `nonlinear_model_details.json`, alongside the selection rankings, inner-selection audits and training-window correlations.

Both legs stay together in validation. The following held-out per-leg performance is descriptive; the recordings are separate sets, so simultaneous bilateral channel relationships cannot be inferred.

In [ ]:
frequency_path = OUT / "selection_frequency.csv"
if frequency_path.exists():
    display(pd.read_csv(frequency_path))
    display(fold_metrics[["test_subject", "selected_features", "best_params", "inner_mean_subject_macro_f1"]])
else:
    print("Majority baseline: no features selected, no coefficients, no inner search.")
display(pd.read_csv(OUT / "per_subject_leg_metrics.csv"))
display(pd.read_csv(OUT / "paired_leg_f1.csv"))
oof = pd.read_csv(OUT / "out_of_fold_predictions.csv")
assert oof.window_id.is_unique and len(oof) == len(windows)
print("Exactly one held-out prediction for every eligible labelled window.")
print("Actual retained feature counts by fold:")
display(fold_metrics[["fold", "test_subject", "n_features", "selected_features", "best_params"]])


## 7. Retain and interpret the RBF results

This model writes its own timestamped `outputs/svm_rbf_selected/<run>/results.csv`, with 15 outer-fold rows, plus all metrics, per-class/per-subject/per-leg tables, predictions, figures and fitted pipelines. The original four-model aggregate `outputs/results.csv` continues to describe those four models; this different, expanded search is saved separately.

If RBF performs better than the original linear SVM, both kernel and feature-search differences are possible explanations. A controlled comparison would require the linear SVM to use the same expanded search, as a separately specified follow-up. Do not revise the RBF grid, selection rule or threshold based on individual outer test outcomes and continue to call them untouched.

The final model is `final_refit/pipeline.joblib`. For future participants, reproduce the same upstream preprocessing/features and call this pipeline without refitting the selector, imputer or scaler on their data. Decision margins are not calibrated probabilities. Load only trusted joblib files.

The next cell saves a results ZIP under the local private outputs directory; no new dataset upload is necessary. Save this executed notebook to retain its displayed outputs.

In [ ]:
import shutil
# Include the descriptive audit in this model's downloadable results.
shutil.copytree(BALANCE_OUT, OUT / "class_balance", dirs_exist_ok=True)
archive = shutil.make_archive(str(OUT.parent / (OUT.name + "_results")), "zip", root_dir=OUT)
print("Results archive:", archive)
print("Save this executed notebook separately to retain displayed outputs.")

print("RBF fold results:", OUT / "results.csv")
print("The original four-model aggregate remains separate.")
